## Connexion 

In [0]:
storage_account_name = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-name")
storage_account_key  = dbutils.secrets.get(scope="energy-bi-scope", key="blob-storage-account-key")
container_name_bronze = "bronze"
container_name_silver = "silver"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)
print("Connexion OK")

## Lire le Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

file_path = f"wasbs://{container_name_bronze}@{storage_account_name}.blob.core.windows.net/energy/household_power_consumption.txt"

df_bronze = spark.read.option("header", "true").option("sep", ",").csv(file_path)

print(f"Bronze lignes brutes : {df_bronze.count()}")

## Nettoyage + Typage

In [0]:
# Remplacer les "?" par null puis caster en double
cols_to_cast = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

df_silver = df_bronze

for col in cols_to_cast:
    df_silver = df_silver.withColumn(
        col, F.when(F.col(col) == "?", None).otherwise(F.col(col).cast(DoubleType()))
    )

# Corriger le type de Time en string propre
df_silver = df_silver.withColumn(
    "Date", F.to_date(F.col("Date"), "d/M/yyyy")
).withColumn("Time", F.col("Time").cast("string"))

# Supprimer les lignes où Date est null
df_silver = df_silver.dropna(subset=["Date"])

print(f"Silver lignes nettoyées : {df_silver.count()}")
df_silver.printSchema()
df_silver.show(5)

## Écrire dans Silver

In [0]:
silver_path = f"wasbs://{container_name_silver}@{storage_account_name}.blob.core.windows.net/energy/"

df_silver.write.mode("overwrite").option("header", "true").csv(silver_path)

print("Données Silver écrites avec succès")